In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import torch
print(torch.cuda.is_available())

True


In [17]:
!nvidia-smi

Sat Mar  7 08:47:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Dataset

### Load Dataset

In [78]:
import pandas as pd
from torchvision import transforms, datasets

file_path = '/content/drive/MyDrive/CPE 313 - Advanced Machine Learning and Deep Learning/Assessment Task 4.1 Supplementary Activity on Transfer Learning/AutismDataset/consolidated'

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

autism_dataset = datasets.ImageFolder(
    root = file_path,
    transform=transform
)

### Dataset Splitting

In [79]:
from torch.utils.data import random_split

train_size = int(0.8 * len(autism_dataset))
test_size = len(autism_dataset) - train_size

train_dataset, test_dataset = random_split(autism_dataset, [train_size, test_size])

### DataLoader

In [80]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    num_workers=2
)

## Model A

### Pre-trained Model (Resnet18)

In [81]:
import torch
import torch.nn as nn
import torchvision.models as models

In [82]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_a = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [83]:
# Finetuning

for param in model_a.parameters():
    param.requires_grad = True

In [84]:
# Final Layer replacement

model_a.fc = nn.Linear(model_a.fc.in_features, 2)

In [85]:
model_a = model_a.to(device)

# Loss Function

criterion = nn.CrossEntropyLoss()

# Optimizer

optimizer = torch.optim.Adam(model_a.parameters(), lr=0.0001)

### Train

In [86]:
num_epochs = 5

for epoch in range(num_epochs):

    model_a.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model_a(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    print(f"Epoch {epoch+1} completed")

Epoch 1 completed
Epoch 2 completed
Epoch 3 completed
Epoch 4 completed
Epoch 5 completed


### Results Evaluation

In [87]:
correct = 0
total = 0

model_a.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model_a(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 84.18367346938776


## Model B

### Model using ConvNet as Feature Extractor

In [88]:
import torch
import torch.nn as nn
import torchvision.models as models

In [89]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_b = models.resnet18(pretrained=True)

In [90]:
# Feature Extraction

for param in model_b.parameters():
    param.requires_grad = False

In [91]:
# Final Layer replacement

model_b.fc = nn.Linear(model_b.fc.in_features, 2)

In [95]:
model_b = model_b.to(device)

# Loss Function

criterion = nn.CrossEntropyLoss()

# Optimizer

optimizer = torch.optim.Adam(model_b.fc.parameters(), lr=0.0001)

### Train

In [96]:
num_epochs = 5

for epoch in range(num_epochs):

    model_b.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model_b(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    print(f"Epoch {epoch+1} completed")

Epoch 1 completed
Epoch 2 completed
Epoch 3 completed
Epoch 4 completed
Epoch 5 completed


### Results Evaluation

In [97]:
correct = 0
total = 0

model_b.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model_b(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 75.68027210884354
